kernel : Python (Pixi)

#### Main Dataset

In [1]:
import anndata
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

anndata.settings.allow_write_nullable_strings = True

series_name = "macnair"
clustered_data_location = find_env_dir("CLUSTERED_DATA")

macnair_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "CLDN11",  
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX5", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "SLC2A5", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_adata, series_name, cell_marker_genes, group = "leiden")

In [20]:
leiden_to_celltype = {
    "0": "Astrocyte",
    "1": "ExcitatoryNeuron",
    "2": "ExcitatoryNeuron",
    "3": "OPC",
    "4": "Pericyte",
    "5": "ExcitatoryNeuron",
    "6": "InhibitoryNeuron",
    "7": "InhibitoryNeuron",
    "8": "InhibitoryNeuron",
    "9": "Endothelial",
    "10": "Oligodendrocyte",
    "11": "Oligodendrocyte",
    "12": "InhibitoryNeuron",
    "13": "Oligodendrocyte",
    "14": "ExcitatoryNeuron",
    "15": "Tcell",
    "16": "ExcitatoryNeuron",
    "17": "Oligodendrocyte",
    "18": "Astrocyte",
    "19": "Oligodendrocyte",
    "20": "Oligodendrocyte",
    "21": "ExcitatoryNeuron",
    "22": "ExcitatoryNeuron",
    "23": "ExcitatoryNeuron",
    "24": "ExcitatoryNeuron",
    "25": "Oligodendrocyte",
    "26": "InhibitoryNeuron",
    "27": "Bcell",
    "28": "InhibitoryNeuron",
    "29": "Microglia",
    "30": "COP",
    "31": "Microglia",
    "32": "ExcitatoryNeuron",
    "33": "ExcitatoryNeuron",
    "34": "Astrocyte",
    "35": "VLMC",
    "36": "Microglia",
    "37": "Oligodendrocyte",
    "38": "ExcitatoryNeuron",
    "39": "Oligodendrocyte",
    "40": "Astrocyte",
    "41": "ExcitatoryNeuron",
    "42": "ExcitatoryNeuron",
    "43": "Oligodendrocyte",
    "44": "Oligodendrocyte",
    "45": "Endothelial",
    "46": "ExcitatoryNeuron",
    "47": "ExcitatoryNeuron",
    "48": "Microglia",
    "49": "Oligodendrocyte",
    "50": "OPC",
    "51": "ExcitatoryNeuron",
    "52": "ExcitatoryNeuron",
    "53": "ExcitatoryNeuron",
    "54": "ExcitatoryNeuron",
    "55": "Doublet",
    "56": "Astrocyte",
    "57": "Astrocyte",
    "58": "Doublet",
    "59": "ExcitatoryNeuron",
    "60": "Astrocyte",
    "61": "Oligodendrocyte",
    "62": "Oligodendrocyte",
    "63": "Ependymal", 
    "64": "Oligodendrocyte",
    "65": "Astrocyte",
    "66": "Doublet",
}

del macnair_adata.obsp

leiden_str = macnair_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_adata.obs["cell"] = mapped

macnair_adata = macnair_adata[macnair_adata.obs["cell"] != "Doublet"].copy()

In [21]:
macnair_adata.write_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [11]:
plot.plot_umap(macnair_adata, series_name, has_celltype=True)

In [22]:
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

opc = macnair_adata[
    (macnair_adata.obs["cell"] == "OPC") |
    (macnair_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

oligodendrocyte = macnair_adata[macnair_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligodendrocyte.h5ad"))

oligo = macnair_adata[
    (macnair_adata.obs["cell"] == "OPC") |
    (macnair_adata.obs["cell"] == "COP") | 
    (macnair_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligo.h5ad"))

#### Validation Dataset

In [ ]:
import anndata
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

anndata.settings.allow_write_nullable_strings = True

series_name = "macnair_validation"
clustered_data_location = find_env_dir("CLUSTERED_DATA")

macnair_validation_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "CLDN11",  
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX5", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "SLC2A5", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_validation_adata, series_name, cell_marker_genes, group = "leiden")

In [37]:
leiden_to_celltype = {
    "0": "Endothelial",
    "1": "Microglia",
    "2": "Astrocyte",
    "3": "Astrocyte",
    "4": "ExcitatoryNeuron",
    "5": "VLMC",
    "6": "Microglia",
    "7": "InhibitoryNeuron",
    "8": "OPC",
    "9": "Pericyte",
    "10": "Astrocyte",
    "11": "Astrocyte",
    "12": "InhibitoryNeuron",
    "13": "Astrocyte",
    "14": "Oligodendrocyte",
    "15": "Ependymal",
    "16": "Ependymal",
    "17": "ExcitatoryNeuron",
    "18": "Tcell",
    "19": "Bcell",
    "20": "InhibitoryNeuron",
    "21": "InhibitoryNeuron",
    "22": "Microglia",
    "23": "Oligodendrocyte",
    "24": "Oligodendrocyte",
    "25": "ExcitatoryNeuron",
    "26": "Oligodendrocyte",
    "27": "Astrocyte",
    "28": "Oligodendrocyte",
    "29": "Oligodendrocyte",
    "30": "COP",
    "31": "ExcitatoryNeuron",
    "32": "Microglia",
    "33": "InhibitoryNeuron",
    "34": "ExcitatoryNeuron",
    "35": "Oligodendrocyte",
    "36": "Doublet",
    "37": "Oligodendrocyte",
    "38": "Oligodendrocyte",
    "39": "Oligodendrocyte",
    "40": "Oligodendrocyte",
    "41": "InhibitoryNeuron",
    "42": "Oligodendrocyte",
    "43": "InhibitoryNeuron",
    "44": "Oligodendrocyte",
    "45": "Oligodendrocyte",
    "46": "Oligodendrocyte",
    "47": "ExcitatoryNeuron",
    "48": "ExcitatoryNeuron",
    "49": "Oligodendrocyte",
    "50": "InhibitoryNeuron",
    "51": "Oligodendrocyte",
    "52": "Oligodendrocyte",
    "53": "Oligodendrocyte",
    "54": "Oligodendrocyte",
    "55": "OPC",
    "56": "ExcitatoryNeuron",
    "57": "Oligodendrocyte",
    "58": "ExcitatoryNeuron",
    "59": "Oligodendrocyte",
    "60": "ExcitatoryNeuron",
    "61": "ExcitatoryNeuron",
    "62": "ExcitatoryNeuron",
    "63": "Doublet", 
    "64": "ExcitatoryNeuron",
    "65": "ExcitatoryNeuron",
    "66": "ExcitatoryNeuron",
}

del macnair_validation_adata.obsp

leiden_str = macnair_validation_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_validation_adata.obs["cell"] = mapped

macnair_validation_adata = macnair_validation_adata[macnair_validation_adata.obs["cell"] != "Doublet"].copy()

In [38]:
macnair_validation_adata.write_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [39]:
plot.plot_umap(macnair_validation_adata, series_name, has_celltype=True)

In [40]:
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

opc = macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "OPC") |
    (macnair_validation_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

oligodendrocyte = macnair_validation_adata[macnair_validation_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligodendrocyte.h5ad"))

oligo= macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "OPC") |
    (macnair_validation_adata.obs["cell"] == "COP") | 
    (macnair_validation_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligo.h5ad"))